In [64]:
from openai import AsyncOpenAI
from dotenv import load_dotenv
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from enum import Enum
from langchain.chat_models import init_chat_model
import asyncio
import langchain

monitor_model = init_chat_model("gpt-4o-mini")
load_dotenv("/mnt/c/Users/jsh27/CSProj/academy-harness/agent-squared/.env")

True

In [60]:
class status(Enum):
    STUCK = 1
    PROGRESSING = 2
    COMPLETED = 3

In [ ]:
class stream_buffer:

    def __init__(self, )->None:
        #(Order, Buffer_text)
        self.buffer: list[tuple[int, str]] = []
        self.counter:int = 0
        self.done: bool = False

    def append(self, append_token:str) -> None:
        self.counter += 1
        self.buffer.append((self.counter, append_token))

    def return_snapshot(self, snapshots_to_return:int) -> list[str]:
        
        last_snapshot:int = self.counter - snapshots_to_return
        returned_buffer = []

        for item in self.buffer:
            if item[0] >= last_snapshot:
                returned_buffer.append(item[1])

        return returned_buffer

In [ ]:

class MonitorState(TypedDict):

    # last 5 summaries
    reasoning_sums: list[str]

    # Compact summary of everything before
    last_sum: str

    # Combined current summary
    curr_sum: str

    curr_analysis:str 
    curr_state:status
    

# Nodes
async def compact_node(state:MonitorState):

    total_sum = "".join(state["reasoning_sums"])

    prompt = f"This is the summary of the earlier part of the reasoning loop: {state['last_sum']} This is the most recent summary of the reasoning loop: {total_sum}. Given all of them, condense it to ~200 words of how its been going so far"

    msg = await monitor_model.ainvoke(prompt)

    return {"curr_sum":msg.content}


async def analyze_node(state:MonitorState):

    prompt = f"Analyze the following compact reasoning summary and generate a ~100 word analysis on how you think its going. This is the reasoning summary: {state['curr_sum']}"

    msg = await monitor_model.ainvoke(prompt)
    return {"curr_analysis":msg.content}



async def classify_node(state:MonitorState):

    prompt = f"given this prev anaylsis, please make one of the following choices on how you think reasoning is going: 'Stuck', 'Progressing', 'Completed'. Please only return one of the three words in exactly that format. {state['curr_analysis']}"

    msg = await monitor_model.ainvoke(prompt)
    return {"curr_state":msg.content}


    
def report_node(state:MonitorState):

    print(state["curr_state"])
    return {}


In [ ]:
workflow = StateGraph(MonitorState)

workflow.add_node("compact_node", compact_node)
workflow.add_node("analyze_node", analyze_node)
workflow.add_node("classify_node", classify_node)
workflow.add_node("report_node", report_node)


workflow.add_edge(START, "compact_node")
workflow.add_edge("compact_node", "analyze_node")
workflow.add_edge("analyze_node", "classify_node")
workflow.add_edge("classify_node", "report_node")
workflow.add_edge("report_node", END)

chain = workflow.compile()

In [ ]:
new_buffer = stream_buffer()
client = AsyncOpenAI()

prompt = """Write a 200 essay about Albert Einstein"""

stream = await client.responses.create(
    model="o4-mini",
    reasoning={"effort":"low","summary":"auto"},
    input=[{"role":"user", "content":prompt}],
    stream=True,

)

In [ ]:
compacted_sum = ""
saved_state = status.PROGRESSING

async for event in stream:
    if event.type=="response.reasoning_summary_text.delta":
        new_buffer.append(event.delta)
        print("Reasoning chunk appended")

        if new_buffer.counter % 5 == 0 and new_buffer.counter != 0:
            recent_sums = new_buffer.return_snapshot(5)
            
            result = await chain.ainvoke({"reasoning_sums":recent_sums, "last_sum":compacted_sum, "curr_sum":"", "curr_analysis":"", "curr_state":saved_state})
        
            compacted_sum = result["curr_sum"]
            saved_state = result["curr_state"]
            